In [2]:
import os, pathlib, math, random, numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds

print("TF:", tf.__version__)
SEED = 1337
tf.random.set_seed(SEED)
np.random.seed(SEED)

# Target words of micro_speech:
WANTED_WORDS = ["yes", "no"]  # giữ giống micro_speech
NUM_CLASSES = 4  # yes, no, unknown, silence

# Audio config (micro_speech expects these):
SAMPLE_RATE = 16000
CLIP_DURATION_MS = 1000
CLIP_SAMPLES = int(SAMPLE_RATE * CLIP_DURATION_MS / 1000)

# Frame config from micro_speech docs:
WINDOW_SIZE_MS = 30
WINDOW_STRIDE_MS = 20

TF: 2.20.0


In [4]:
# Speech Commands v0.02 is commonly used; TFDS provides speech_commands.
# It will download to ~/tensorflow_datasets by default.
ds_train, ds_val, ds_test = tfds.load(
    "speech_commands",
    split=["train", "validation", "test"],
    as_supervised=True,   # returns (audio, label)
    with_info=False
)

print(ds_train)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/3 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling C:\Users\nguye\tensorflow_datasets\speech_commands\incomplete.73AY9I_0.0.3\speech_commands-train.tfr…

Generating validation examples...: 0 examples [00:00, ? examples/s]

Shuffling C:\Users\nguye\tensorflow_datasets\speech_commands\incomplete.73AY9I_0.0.3\speech_commands-validatio…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling C:\Users\nguye\tensorflow_datasets\speech_commands\incomplete.73AY9I_0.0.3\speech_commands-test.tfre…

Dataset speech_commands downloaded and prepared to C:\Users\nguye\tensorflow_datasets\speech_commands\0.0.3. Subsequent calls will reuse this data.
<_PrefetchDataset element_spec=(TensorSpec(shape=(None,), dtype=tf.int16, name=None), TensorSpec(shape=(), dtype=tf.int64, name=None))>


In [5]:
# TFDS label is integer. Let's get label names:
label_names = tfds.builder("speech_commands").info.features["label"].names
label_to_name = {i:n for i,n in enumerate(label_names)}

TARGET_SET = set(WANTED_WORDS)

# We'll map to: 0=yes, 1=no, 2=unknown, 3=silence
NAME_TO_CLASS = {"yes":0, "no":1, "unknown":2, "silence":3}

def normalize_audio(audio):
    # audio from TFDS is int16 PCM; convert to float32 [-1, 1]
    audio = tf.cast(audio, tf.float32) / 32768.0
    return audio

def pad_or_trim(audio, desired_samples=CLIP_SAMPLES):
    audio = audio[:desired_samples]
    pad_len = desired_samples - tf.shape(audio)[0]
    audio = tf.cond(
        pad_len > 0,
        lambda: tf.pad(audio, [[0, pad_len]]),
        lambda: audio
    )
    return audio

def map_to_4class(audio, label_id):
    name = tf.gather(label_names, label_id)
    # name is tf.string
    # If name is yes/no -> target; else -> unknown
    cls = tf.cond(
        tf.reduce_any(tf.equal(name, list(TARGET_SET))),
        lambda: tf.where(tf.equal(name, "yes"), 0, 1),
        lambda: tf.constant(NAME_TO_CLASS["unknown"], dtype=tf.int32)
    )
    audio = normalize_audio(audio)
    audio = pad_or_trim(audio)
    return audio, cls

In [6]:
def make_silence_dataset(num_samples):
    # float32 zeros shape (CLIP_SAMPLES,)
    audios = tf.zeros([num_samples, CLIP_SAMPLES], dtype=tf.float32)
    labels = tf.fill([num_samples], tf.constant(NAME_TO_CLASS["silence"], dtype=tf.int32))
    return tf.data.Dataset.from_tensor_slices((audios, labels))

# Add silence about ~10% of target size (tùy bạn chỉnh)
SILENCE_TRAIN = 3000
SILENCE_VAL   = 500
SILENCE_TEST  = 500

ds_train_4 = ds_train.map(map_to_4class, num_parallel_calls=tf.data.AUTOTUNE)
ds_val_4   = ds_val.map(map_to_4class, num_parallel_calls=tf.data.AUTOTUNE)
ds_test_4  = ds_test.map(map_to_4class, num_parallel_calls=tf.data.AUTOTUNE)

ds_train_4 = ds_train_4.concatenate(make_silence_dataset(SILENCE_TRAIN))
ds_val_4   = ds_val_4.concatenate(make_silence_dataset(SILENCE_VAL))
ds_test_4  = ds_test_4.concatenate(make_silence_dataset(SILENCE_TEST))

In [7]:
# Compute spectrogram with window/stride matching micro_speech
frame_length = int(SAMPLE_RATE * WINDOW_SIZE_MS / 1000)   # 480
frame_step   = int(SAMPLE_RATE * WINDOW_STRIDE_MS / 1000) # 320

NUM_MEL_BINS = 40
LOWER_EDGE_HZ = 20.0
UPPER_EDGE_HZ = 4000.0

def audio_to_feature(audio):
    # audio: float32 [CLIP_SAMPLES]
    stft = tf.signal.stft(
        audio,
        frame_length=frame_length,
        frame_step=frame_step,
        fft_length=512
    )  # [frames, fft_bins]
    spectrogram = tf.abs(stft) ** 2

    num_spectrogram_bins = spectrogram.shape[-1]
    mel_w = tf.signal.linear_to_mel_weight_matrix(
        num_mel_bins=NUM_MEL_BINS,
        num_spectrogram_bins=num_spectrogram_bins,
        sample_rate=SAMPLE_RATE,
        lower_edge_hertz=LOWER_EDGE_HZ,
        upper_edge_hertz=UPPER_EDGE_HZ
    )
    mel = tf.matmul(spectrogram, mel_w)  # [frames, mel_bins]
    log_mel = tf.math.log(mel + 1e-6)

    # Ensure fixed time dimension 49
    # For 1s audio with these params, frames should be 49
    log_mel = log_mel[:49, :]
    pad_t = 49 - tf.shape(log_mel)[0]
    log_mel = tf.cond(pad_t > 0, lambda: tf.pad(log_mel, [[0,pad_t],[0,0]]), lambda: log_mel)

    # Add channel dim -> (49, 40, 1)
    return tf.expand_dims(log_mel, axis=-1)

def add_features(audio, label):
    feat = audio_to_feature(audio)
    return feat, label

In [8]:
BATCH_SIZE = 64

train_ds = (ds_train_4
            .shuffle(20000, seed=SEED)
            .map(add_features, num_parallel_calls=tf.data.AUTOTUNE)
            .batch(BATCH_SIZE)
            .prefetch(tf.data.AUTOTUNE))

val_ds = (ds_val_4
          .map(add_features, num_parallel_calls=tf.data.AUTOTUNE)
          .batch(BATCH_SIZE)
          .prefetch(tf.data.AUTOTUNE))

test_ds = (ds_test_4
           .map(add_features, num_parallel_calls=tf.data.AUTOTUNE)
           .batch(BATCH_SIZE)
           .prefetch(tf.data.AUTOTUNE))

# Quick check:
for x, y in train_ds.take(1):
    print("Feature batch:", x.shape, x.dtype)
    print("Label batch:", y.shape, y.dtype)

Feature batch: (64, 49, 40, 1) <dtype: 'float32'>
Label batch: (64,) <dtype: 'int32'>


In [9]:
def build_model():
    inputs = tf.keras.Input(shape=(49, 40, 1))
    x = tf.keras.layers.Conv2D(8, (3,3), padding="same", activation="relu")(inputs)
    x = tf.keras.layers.MaxPool2D((2,2))(x)
    x = tf.keras.layers.Conv2D(16, (3,3), padding="same", activation="relu")(x)
    x = tf.keras.layers.MaxPool2D((2,2))(x)
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(32, activation="relu")(x)
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = tf.keras.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

model = build_model()
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 49, 40, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 49, 40, 8)      │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 24, 20, 8)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 24, 20, 16)     │         1,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 12, 10, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1920)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │        61,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 62,852 (245.52 KB)

 Trainable params: 62,852 (245.52 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
EPOCHS = 10

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=2, restore_best_weights=True)
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

test_loss, test_acc = model.evaluate(test_ds)
print("Test acc:", test_acc)

Epoch 1/10
 129/1711 ━━━━━━━━━━━━━━━━━━━━ 21:39 821ms/step - accuracy: 0.9659 - loss: 0.1013